# EDA: UA Locations Extraction

## What we are looking for

Input: a text in Ukrainian, Russian or mixed code, usually a Telegram post. Output: a list of location strings exactly as written in the text. Locations are cities, countries, regions, districts, streets and addresses, and other geographic objects. For example, `Сьогодні з Києва до Хелма вперше вирушив новий поїзд` should return `["Києва", "Хелма"]`. Scoring is F1 over sets of strings.

The model must be recurrent (GRU/LSTM), so the task is token tagging with O/B/I tags that are converted back into strings.

## What we work with

| File | What it is | Size | Labels |
|---|---|---|---|
| `uk_geo_dataset.csv` | news sentences, Ukr-Synth corpus | 1.01M | machine LOC/ORG/PER spans |
| `ru_geo_dataset.csv` | Lenta.ru news sentences, Nerus corpus | 8.03M | machine LOC/ORG/PER spans |
| `uk_geo_dataset_processed_v1.parquet` | tokens of the same uk corpus | 1.01M | flat O/LOC, no B/I |
| `labeling_sample.csv` | Telegram posts | 26 | manual |
| `test.csv` | Telegram posts | 477 | none |

Train is large, but it is news with machine labels. We are scored on Telegram, and only 26 posts have manual labels.

This notebook trains no models. It checks ideas from papers and competitions against the data and ends with pointers for experiments.

In [1]:
import ast, json, re, string, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")
pd.set_option("display.max_colwidth", 120, "display.max_rows", 100)

In [2]:
# Official metric, copied verbatim from the competition page (including the non-f-string regex).
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\d", "", text)
    text = re.sub(r"\b\w\b\s?", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()


def comp_metric(y_true, y_pred):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p = 0.0, 0.0, 0.0, 0.0
    for y_true_sample, y_pred_sample in zip(y_true, y_pred):
        y_true_sample = {hard_processing(s) for s in y_true_sample}
        y_pred_sample = {hard_processing(s) for s in y_pred_sample}
        tp += len(y_true_sample & y_pred_sample)
        fp += len(y_pred_sample - y_true_sample)
        fn += len(y_true_sample - y_pred_sample)
        p += len(y_true_sample)
    precision = tp / (tp + fp) if tp + fp else float(p == 0)
    recall = tp / (tp + fn) if tp + fn else float(p == 0)
    return round(2 * precision * recall / (precision + recall), 8) if precision + recall else 0

In [3]:
# Tokens and BIO. Hyphens are separate tokens, so a label can stop at one ("Київ-Одеса" is two locations).
TOKEN = re.compile(r"\w+(?:['’ʼ]\w+)*|[^\w\s]")
SPAN = re.compile(r"\((\d+),\s*(\d+)\)")
BOUND_PREFIXES = {"івано", "нью", "санкт", "києво", "кам'янці", "кам'янець", "куала", "рава", "конча", "кончі", "австро", "ростові", "переяславі"}


def parse_spans(value):
    return [(int(a), int(b)) for a, b in SPAN.findall(str(value))]


def tokenize(text):
    return [m.span() for m in TOKEN.finditer(text)]


def clean_spans(text, spans):
    # Joins names split at a hyphen, drops spans without letters, trims edge whitespace.
    merged = []
    for a, b in sorted(spans):
        if merged:
            gap = text[merged[-1][1]:a]
            if a <= merged[-1][1] or gap == "" or (gap == "-" and text[merged[-1][0]:merged[-1][1]].lower() in BOUND_PREFIXES):
                merged[-1] = (merged[-1][0], max(b, merged[-1][1]))
                continue
        merged.append((a, b))
    cleaned = []
    for a, b in merged:
        chunk = text[a:b]
        a, b = a + len(chunk) - len(chunk.lstrip()), b - (len(chunk) - len(chunk.rstrip()))
        if a < b and any(c.isalnum() for c in text[a:b]):
            cleaned.append((a, b))
    return cleaned


def spans_to_bio(offsets, spans):
    tags, misaligned = ["O"] * len(offsets), 0
    for a, b in spans:
        hit = [i for i, (s, e) in enumerate(offsets) if s < b and a < e]
        if not hit or offsets[hit[0]][0] != a or offsets[hit[-1]][1] != b:
            misaligned += 1
        if hit:
            tags[hit[0]] = "B"
            for i in hit[1:]:
                tags[i] = "I"
    return tags, misaligned


def bio_to_strings(text, offsets, tags, merge_adjacent=False):
    out, i = [], 0
    while i < len(tags):
        if tags[i] == "O":
            i += 1
            continue
        j = i + 1
        while j < len(tags) and (tags[j] == "I" or (merge_adjacent and tags[j] != "O")):
            j += 1
        out.append(text[offsets[i][0]:offsets[j - 1][1]])
        i = j
    return out

## 1. Data and its quality

Question: what can go into training as is, what must be cleaned, and is there any leakage?

In [4]:
uk = pd.read_csv(DATA / "uk_geo_dataset.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "labeling_sample.csv")
for col in ["loc_markers", "org_markers"]:
    uk[col] = uk[col].map(parse_spans)
sample["loc_markers"] = sample["loc_markers"].map(parse_spans)
sample["locations"] = sample["locations"].map(ast.literal_eval)

# ru has 8M rows: hash all of them, keep a 5% random sample for label statistics and training.
rng = np.random.default_rng(0)
ru_parts, ru_hashes, ru_docs = [], [], []
for chunk in pd.read_csv(DATA / "ru_geo_dataset.csv", usecols=["text", "loc_markers", "doc_id"], chunksize=500_000):
    ru_hashes.append(pd.util.hash_array(chunk["text"].to_numpy(dtype=object)))
    ru_docs.append(chunk["doc_id"].to_numpy())
    ru_parts.append(chunk[rng.random(len(chunk)) < 0.05])
ru = pd.concat(ru_parts, ignore_index=True)
ru["loc_markers"] = ru["loc_markers"].map(parse_spans)
ru_hashes, ru_docs = np.concatenate(ru_hashes), np.concatenate(ru_docs)

uk["spans"] = [clean_spans(t, s) for t, s in zip(uk["text"], uk["loc_markers"])]
ru["spans"] = [clean_spans(t, s) for t, s in zip(ru["text"], ru["loc_markers"])]
sample["spans"] = sample["loc_markers"]
{"uk": uk.shape, "ru_rows_total": len(ru_hashes), "ru_sample": ru.shape, "test": test.shape, "labeling_sample": sample.shape}

{'uk': (1010000, 6),
 'ru_rows_total': 8028840,
 'ru_sample': (402053, 4),
 'test': (477, 3),
 'labeling_sample': (26, 4)}

In [5]:
# One real example per source, with entities in [brackets].
def mark(text, spans):
    out, pos = [], 0
    for a, b in sorted(spans):
        out += [text[pos:a], "[", text[a:b], "]"]
        pos = b
    return "".join(out) + text[pos:]


for name, df in {"uk": uk, "ru": ru, "labeling_sample": sample}.items():
    row = df[df["spans"].map(len).ge(2)].iloc[3]
    print(f"{name}:\n{mark(row['text'], row['spans'])[:350]}\n")
print("test (no labels):\n", test["text"].iloc[4][:350])

uk:
Кількість життів, які доведеться віддати [Україні] за наступне існування у вигляді провінцій-сателітів [Румунії], [Польщі], [Росії] неможливо спрогнозувати - економічне прогнозування цим не займається.

ru:
В нем террористы указали, что нападавший «был солдатом ИГ и провел операцию в ответ на призывы атаковать граждан стран международной коалиции», которая проводит операции в [Сирии] и [Ираке].

labeling_sample:
Між [Києвом] та [Васильковом] “Укрзалізниця” запустила додаткові приміські поїзди.

🚉В АТ "Укрзалізниця" призначили додаткові прискорені приміські поїзди сполученням між Васильковом та Києвом.

✔️ Курсуватимуть електрички щодня, крім вихідних, до кінця вересня.

🚊Відправлення з Приміського вокзалу Київ-Пасажирський та прибуття на станцію Васильків-

test (no labels):
 ⛸В один день, 29 серпня, ДП "Центральна учбово-тренувальна база по ковзанярському спорту "Льодовий стадіон" оголосило та одразу ж відмінило тендер вартістю 307 млн гривень на добудову Національного тенісного ц

In [6]:
normalize_text = lambda s: re.sub(r"\W+", " ", s.lower()).strip()
uk["norm"] = uk["text"].map(normalize_text)
train_norm = set(uk.loc[uk["is_valid"].eq(0), "norm"])
uk["leak"] = uk["is_valid"].eq(1) & uk["norm"].isin(train_norm)
uk["tch_footer"] = uk["text"].str.contains(r"TCH\.?\s*$")
valid_mask = uk["is_valid"].eq(1) & ~uk["leak"] & ~uk["tch_footer"]


def marker_defects(df):
    c = Counter()
    for text, spans in zip(df["text"], df["loc_markers"]):
        for a, b in spans:
            s = text[a:b]
            c["spans"] += 1
            c["no_letters_or_digits"] += not any(ch.isalnum() for ch in s)
            c["edge_space_or_punct"] += not (s[0].isalnum() and s[-1].isalnum())
            c["mid_word_boundary"] += (a > 0 and text[a - 1].isalnum() and s[0].isalnum()) or (b < len(text) and text[b].isalnum() and s[-1].isalnum())
    return c


parquet_labels = pd.read_parquet(DATA / "uk_geo_dataset_processed_v1.parquet", columns=["labels"])["labels"].head(5000)
parquet_tags = Counter(tag for v in parquet_labels for tag in (json.loads(v) if isinstance(v, str) else v))
defects = pd.DataFrame({"uk": marker_defects(uk), "ru_sample": marker_defects(ru), "sample": marker_defects(sample)}).fillna(0).astype(int)
print(pd.Series({
    "nulls in text/markers (uk)": int(uk[["text", "loc_markers"]].isna().sum().sum()),
    "uk exact duplicate rows": int(uk["text"].duplicated().sum()),
    "uk normalised duplicate rows": int(uk["norm"].duplicated().sum()),
    "uk is_valid rows also in train": int(uk["leak"].sum()),
    "uk rows with TCH section footer": int(uk["tch_footer"].sum()),
    "uk LOC spans in TCH rows": int(uk.loc[uk["tch_footer"], "loc_markers"].map(len).sum()),
    "uk spans removed by cleaning": int(uk["loc_markers"].map(len).sum() - uk["spans"].map(len).sum()),
    "ru exact duplicate rows": int(len(ru_hashes) - len(np.unique(ru_hashes))),
    "test texts found in uk": int(test["text"].map(normalize_text).isin(set(uk["norm"])).sum()),
}).to_string())
print("parquet tags:", dict(parquet_tags))
defects

nulls in text/markers (uk)              0
uk exact duplicate rows              2454
uk normalised duplicate rows         3786
uk is_valid rows also in train         79
uk rows with TCH section footer      9249
uk LOC spans in TCH rows             9714
uk spans removed by cleaning         3942
ru exact duplicate rows            523501
test texts found in uk                  0
parquet tags: {'O': 86060, 'LOC': 2156}


,uk,ru_sample,sample
spans,345114,185044,32
no_letters_or_digits,1057,4,0
edge_space_or_punct,5364,824,0
mid_word_boundary,1516,385,1


**Findings**
- There are no missing values and all spans are in bounds. Train and test do not overlap.
- The prepared parquet cannot be used: its tags are flat O/LOC, so adjacent locations merge. BIO has to be rebuilt from `loc_markers`.
- Remove before training: 79 `is_valid` rows that also appear in train (leakage), duplicates (uk 3,786 normalised, ru 523,501 exact), and 9,249 rows ending in the TCH section footer (9,714 spans, including `Укрaїнa` with a Latin `a` and `Світ`).
- Cleaning changes 3,942 uk spans, mostly split names such as `Івано` + `-Франківськ` and spans that are only a hyphen.

## 2. The target

The target is the set of location strings in a text. Question: what counts as a location in the silver train and in the manual sample, and where do the two policies differ?

In [7]:
STREET = r"(?i)\bвул|\bпроспект|\bбульвар|\bпров\.|\bпровул|\bплощ|\bузвіз|\bнабережн|\bулиц|\bпереул|\bплощад"
DISTRICT = r"(?i)район|р-н|област|\bобл\b|щин[аіиу]"
POI = r"\b(?:ТЦ|ТРЦ|ЖК|метро|Метро)\b"


def entity_table(df):
    per_text = df["spans"].map(len)
    strings = pd.Series([t[a:b] for t, s in zip(df["text"], df["spans"]) for a, b in s], dtype=str)
    return {
        "texts": len(df),
        "no LOC, share of texts": round((per_text == 0).mean(), 3),
        "entities per text, mean": round(per_text.mean(), 2),
        "entities per text with LOC, max": int(per_text.max()),
        "one-word entities": round((strings.str.split().str.len() == 1).mean(), 3),
        "street/address": round(strings.str.contains(STREET).mean(), 3),
        "district/region": round(strings.str.contains(DISTRICT).mean(), 3),
        "POI (ТЦ, ЖК, метро)": round(strings.str.contains(POI).mean(), 3),
        "street/address entities, count": int(strings.str.contains(STREET).sum()),
        "Україн*/Украин* (noun)": round(strings.str.fullmatch(r"(?i)украин\w*|україн\w*").mean(), 3),
    }


pd.DataFrame({"uk": entity_table(uk), "ru_sample": entity_table(ru), "labeling_sample": entity_table(sample)})

,uk,ru_sample,labeling_sample
texts,1010000.000,402053.000,26.000
"no LOC, share of texts",0.769,0.694,0.385
"entities per text, mean",0.340,0.460,1.230
"entities per text with LOC, max",21.000,33.000,10.000
one-word entities,0.844,0.881,0.719
street/address,0.016,0.009,0.156
district/region,0.068,0.029,0.125
"POI (ТЦ, ЖК, метро)",0.000,0.000,0.031
"street/address entities, count",5519.000,1603.000,5.000
Україн*/Украин* (noun),0.160,0.037,0.000


In [8]:
# Does the type word ("вул.", "місті") go inside the span? Decides what the tagger must learn at the boundary.
STREET_WORD = re.compile(r"(?i)^(вул|вулиц\w*|ул|улиц\w*|проспект\w*|просп|площ\w*)$")
SETTLEMENT_WORD = re.compile(r"(?i)^(м|місто|міста|місті|село|села|селі|смт|город\w*|поселк\w*)$")
PREV_WORD = re.compile(r"(\w+)\.?\s*$")
FIRST_WORD = re.compile(r"\w+")


def type_word_position(df):
    c = Counter()
    for text, spans in zip(df["text"], df["spans"]):
        for a, b in spans:
            first = FIRST_WORD.search(text[a:b])
            prev = PREV_WORD.search(text[max(0, a - 20):a])
            first, prev = first.group() if first else "", prev.group(1) if prev else ""
            for name, pattern in [("street", STREET_WORD), ("settlement", SETTLEMENT_WORD)]:
                c[f"{name}: inside span"] += bool(pattern.match(first))
                c[f"{name}: before span"] += bool(pattern.match(prev))
    return c


pd.DataFrame({"uk": type_word_position(uk), "ru_sample": type_word_position(ru), "labeling_sample": type_word_position(sample)}).fillna(0).astype(int)

,uk,ru_sample,labeling_sample
street: inside span,4655,120,5
street: before span,655,965,0
settlement: inside span,358,11,0
settlement: before span,8320,4307,0


In [9]:
# How consistently are the most test-relevant strings labelled? Share of whole-word occurrences.
def label_rates(df, pattern, org=True):
    rx = re.compile(pattern)
    c = Counter()
    org_col = df["org_markers"] if org and "org_markers" in df else [[]] * len(df)
    for text, spans, orgs in zip(df["text"], df["spans"], org_col):
        for m in rx.finditer(text):
            a, b = m.span()
            c["occurrences"] += 1
            if any(s <= a and b <= e for s, e in spans):
                c["LOC"] += 1
            elif any(s <= a and b <= e for s, e in orgs):
                c["inside ORG"] += 1
            else:
                c["unlabelled"] += 1
    return {k: (round(v / c["occurrences"], 3) if k != "occurrences" else v) for k, v in c.items()}


UKRAINE = r"\bУкраїн(?:а|и|і|у|ою)\b"
KYIV = r"\b(?:Київ|Києва|Києві|Києву|Києвом)\b"
pd.DataFrame({
    "uk: Україна": label_rates(uk, UKRAINE),
    "sample: Україна": label_rates(sample, UKRAINE, org=False),
    "uk: Київ": label_rates(uk, KYIV),
    "sample: Київ": label_rates(sample, KYIV, org=False),
}).fillna(0)

,uk: Україна,sample: Україна,uk: Київ,sample: Київ
occurrences,75065.000,4.0,15467.000,15.0
inside ORG,0.171,0.0,0.065,0.0
LOC,0.738,0.0,0.925,0.4
unlabelled,0.091,1.0,0.011,0.6


In [10]:
# How often do test posts mention what train barely labels? Occurrences per 1000 texts.
PATTERNS = {"Київ*": r"Київ|Києв", "район": r"район", "вул.": r"вул\.", "вулиц": r"вулиц", "проспект": r"проспект", "метро": r"метро", "ЖК/ТЦ": r"\b(?:ЖК|ТЦ)\b", "Україн*": r"Україн"}
pd.DataFrame({
    name: {k: round(1000 * df["text"].str.contains(p).mean(), 1) for k, p in PATTERNS.items()}
    for name, df in {"uk": uk, "ru_sample": ru, "test": test, "labeling_sample": sample}.items()
})

,uk,ru_sample,test,labeling_sample
Київ*,20.1,0.0,383.6,461.5
район,11.3,11.7,134.2,115.4
вул.,1.2,0.0,39.8,76.9
вулиц,5.7,0.0,44.0,38.5
проспект,0.5,1.0,23.1,38.5
метро,2.0,13.3,14.7,76.9
ЖК/ТЦ,0.1,0.3,14.7,76.9
Україн*,78.1,0.0,155.1,115.4


**Findings**
- Train and sample have different class mixes. Streets and addresses are 1.6% of uk locations but 15.6% in the sample. Train is country-heavy and test is about Kyiv: `Київ*` appears in 38% of test posts and 2% of uk sentences.
- uk still holds 5,519 street spans, so raising their share with existing sentences is a cheap first step.
- The type word sits inside the span in uk and the sample (`вул.`: 4,655 inside, 655 before). In ru it is the opposite (120 inside, 965 before), so raw ru spans teach the wrong boundary for streets.
- `Україна` is labelled in 74% of uk occurrences and in 0 of 4 in the sample. `Київ` is labelled in only 40% of sample mentions, because channel signatures and institution names are not marked. The sample is small, but this is the largest false-positive risk.
- The sample has errors (`вул. Здолбунівська, 1` instead of `17`, skipped stations), so it is not exhaustive labelling.

## 3. The metric

Question: what does the metric reward, where does it break, and what do typical mistakes cost?

In [11]:
cases = ["Київ", "м. Київ", "м.Київ", "Київ,", "Київ.", "(Київ)", "«Київ»", "#Київ", "Київ🫶", "Івано-Франківськ", "Івано - Франківськ",
         "вул. Солом'янська, 28", "вул. Солом'янська", "р-н Оболонь", "с. Гора", "Kyiv", "KYIV", "Odesa", "ODESA", "Nature reserve", "В", "28"]
pd.DataFrame({"raw": cases, "hard_processing": [hard_processing(c) for c in cases]})

,raw,hard_processing
0,Київ,київ
1,м. Київ,київ
2,м.Київ,мкиїв
3,"Київ,","київ,"
4,Київ.,київ
5,(Київ),київ
6,«Київ»,«київ»
7,#Київ,#київ
8,Київ🫶,київ🫶
9,Івано-Франківськ,івано-франківськ


In [12]:
# Proposed alternative: same set logic, correct normalisation, partial credit for boundary errors.
def fixed_norm(s):
    s = unicodedata.normalize("NFKC", s).replace("’", "'").replace("ʼ", "'").lower()
    s = re.sub(r"[^\w\s']|\d", " ", s)
    s = re.sub(r"\b\w\b", " ", s)
    return " ".join(s.split())


def overlap_f1(y_true, y_pred):
    credit = n_true = n_pred = 0
    for t, p in zip(y_true, y_pred):
        t, p = {fixed_norm(s) for s in t} - {""}, {fixed_norm(s) for s in p} - {""}
        n_true, n_pred = n_true + len(t), n_pred + len(p)
        free = set(t)
        for s in sorted(p, key=len, reverse=True):
            score = lambda g: len(set(s.split()) & set(g.split())) / len(set(s.split()) | set(g.split()))
            best = max(free, key=score, default=None)
            if best is not None and score(best) > 0:
                credit += score(best)
                free.remove(best)
    return round(2 * credit / (n_true + n_pred), 4) if n_true + n_pred else 1.0


y_true = sample["locations"].tolist()
perturbations = {
    "perfect": y_true,
    "nothing predicted": [[] for _ in y_true],
    "lowercase": [[s.lower() for s in t] for t in y_true],
    "each string twice": [t + t for t in y_true],
    "prefix 'м. '": [["м. " + s for s in t] for t in y_true],
    "first word only": [[s.split()[0] for s in t] for t in y_true],
    "trailing comma": [[s + "," for s in t] for t in y_true],
    "one extra FP per text": [t + ["Марс"] for t in y_true],
}
pd.DataFrame({
    "official": {k: comp_metric(y_true, v) for k, v in perturbations.items()},
    "alternative": {k: overlap_f1(y_true, v) for k, v in perturbations.items()},
})

,official,alternative
perfect,1.000000,1.0000
nothing predicted,0.000000,0.0000
lowercase,1.000000,1.0000
each string twice,1.000000,1.0000
prefix 'м. ',1.000000,1.0000
first word only,0.718750,0.8594
trailing comma,0.000000,1.0000
one extra FP per text,0.711111,0.7111


In [13]:
# Leaderboard noise: resample texts with sample's entity counts under a model with 70% recall.
# Assumed model quality, not a measurement; public size 224 assumes the 47% split of the 2025 edition.
rng = np.random.default_rng(0)
gold_counts = np.array([len({hard_processing(s) for s in t}) for t in y_true])
top_share = gold_counts.max() / gold_counts.sum()


def simulate(n_texts, recall=0.7, fp_per_entity=0.3, fp_empty=0.05):
    k = rng.choice(gold_counts, n_texts)
    tp = rng.binomial(k, recall)
    fp = rng.poisson(fp_per_entity * (k > 0) + fp_empty)
    return 2 * tp.sum() / max(2 * tp.sum() + fp.sum() + (k - tp).sum(), 1)


print(f"sample: {len(y_true)} texts, {gold_counts.sum()} unique strings, largest text = {top_share:.0%} of them")
pd.DataFrame({n: pd.Series([simulate(n) for _ in range(4000)]).describe(percentiles=[0.025, 0.975])[["mean", "std", "2.5%", "97.5%"]] for n in [26, 224, 477]}).round(3)

sample: 26 texts, 32 unique strings, largest text = 31% of them


,26,224,477
mean,0.735,0.740,0.740
std,0.068,0.022,0.015
2.5%,0.588,0.694,0.709
97.5%,0.852,0.780,0.769


**Strengths.** The metric is simple, needs no character offsets, ignores case and repeats, and matches the real output of the task (a list of strings).

**Weaknesses and edge cases**
- The regex is not an f-string, so punctuation is barely removed. A trailing comma destroys a match, so boundaries must be cleaned: `Київ,` versus `Київ` scores 0.
- The same regex deletes some lowercase Latin letters: `Odesa` becomes `od`, while `ODESA` stays `odesa`. Latin names depend on case.
- There is no lemmatisation: `Київ` and `Києві` are different strings, so we must return the surface form.
- A boundary error costs a false positive and a false negative at once (first word only gives 0.72 on the sample).
- A string of digits or a single letter normalises to an empty string and is still counted. Such predictions must be dropped.
- It is micro-averaged: one list post provides 31% of all sample strings, and any false positive on a text without locations is fully penalised.

**Complementary metrics for experiments:** token-level F1 for B/I (shows whether the model learns), exact span F1 (seqeval), recall by entity type (city, street, district), the share of location-free texts with a false positive, recall on unseen entities.

**Alternative target metric: `overlap_f1`.** The same set-based F1, with correct normalisation (all punctuation, apostrophes, no damage to Latin letters) and partial credit by token Jaccard. A trailing comma no longer zeroes the score (0 versus 1.0), and a boundary error is penalised proportionally (0.72 versus 0.86). This better reflects usefulness, since `вул. Солом'янська` and `вул. Солом'янська, 28` are the same address.

**Leaderboard noise.** Assuming 70% recall and 224 public texts (the 47% split of the 2025 edition), the 95% interval of F1 is about 0.69 to 0.78. On the 26 sample texts it is 0.59 to 0.85, so differences of a few hundredths cannot be seen there.

## 4. Groups, time and validation

Question: which group or time factors exist, and what validation do they call for? Train-versus-test separability is measured in `adversarial_validation.ipynb`.

In [14]:
# Group factors: there is no timestamp or channel column, so channels and reposts are estimated from the text.
def shingles(text, k=4):
    words = TOKEN.findall(text.lower())
    return {tuple(words[i:i + k]) for i in range(max(len(words) - k + 1, 1))}


FOOTER_RE = re.compile(r"(?i)підписат(ись|ися)|подписаться|надіслати новину|запропонувати новину|пряма трансляція|життя🫶оболонь|^@\w+$")
is_footer = lambda line: len(line) < 80 and bool(FOOTER_RE.search(line))
sets = test["text"].map(shingles).tolist()
near = [(i, j) for i in range(len(sets)) for j in range(i + 1, len(sets)) if len(sets[i] & sets[j]) / max(len(sets[i] | sets[j]), 1) >= 0.8]
footers = Counter(line.strip() for t in test["text"] for line in set(t.split("\n")) if is_footer(line.strip()))
sents_per_doc = pd.Series(ru_docs).value_counts()
print(pd.Series({
    "test: near-duplicate pairs (Jaccard >= 0.8)": len(near),
    "test: texts in those pairs": len({i for pair in near for i in pair}),
    "test: posts with a channel footer": int(test["text"].map(lambda t: any(is_footer(l.strip()) for l in t.split("\n"))).sum()),
    "test: text_id strictly increasing": bool(test["text_id"].is_monotonic_increasing),
    "ru: documents": len(sents_per_doc),
    "ru: sentences per doc, median": float(sents_per_doc.median()),
}).to_string())
footers.most_common(10)

test: near-duplicate pairs (Jaccard >= 0.8)         8
test: texts in those pairs                         13
test: posts with a channel footer                 142
test: text_id strictly increasing                True
ru: documents                                  739190
ru: sentences per doc, median                    10.0


[('👉👉Запропонувати новину👈👈', 24),
 ('Підписатись на Дарниця Livе', 24),
 ('Хуйовий Київ | Надіслати новину', 20),
 ('Київ — пряма трансляція 👈', 18),
 ('🔷\xa0Підписатися на Telegram\xa0| Instagram\xa0| Facebook\xa0| TikTok', 18),
 ('Життя🫶Оболонь', 17),
 ('Надіслати новину', 17),
 ('Підписатися\xa0\xa0\xa0 |\xa0\xa0\xa0 Запропонувати новину', 13),
 ('@lossolomas_kyiv', 7),
 ('🔷 Підписатися на Telegram | Instagram | Facebook | TikTok', 6)]

**Findings**
- There is no time information: no dates in the data, and `text_id` only increases. A time-based split is impossible.
- Group factors: `doc_id` in ru (739,190 documents, median 10 sentences), normalised-text duplicates in uk, and in test the channel signatures (142 posts with a footer) and 8 near-duplicate pairs.

**Proposed validation**
1. The same folds for every experiment, grouped by normalised text (uk) and `doc_id` (ru). Silver folds show whether the model learns.
2. `labeling_sample` as a sanity check only, over several seeds, with no threshold tuning on it.
3. Policy decisions (streets, `Україна`, signatures) are tested only by submission.

Correlation between CV and the leaderboard is expected to be weak because of the domain gap and the different labelling policy. It should be higher for tokenisation and cleaning changes, and lower for anything that depends on the share of streets.

## 5. Sequences for the RNN

Question: how long are the inputs, what is lost in tokenisation, and which context signals actually carry information?

In [15]:
length = lambda df: df["text"].map(lambda t: len(TOKEN.findall(t)))
lengths = {"uk": length(uk.sample(200_000, random_state=0)), "ru_sample": length(ru), "test": length(test), "labeling_sample": length(sample)}
table = pd.DataFrame({k: v.describe(percentiles=[0.5, 0.9, 0.99])[["50%", "90%", "99%", "max"]] for k, v in lengths.items()}).round(0).astype(int)
EMOJI = "[\U0001F000-\U0001FAFF☀-➿]"
shape = lambda df: {
    "multi-line texts": round(df["text"].str.contains("\n").mean(), 3),
    "texts with emoji": round(df["text"].str.contains(EMOJI).mean(), 3),
    "emoji glued to a word": round(df["text"].str.contains(rf"\w{EMOJI}|{EMOJI}\w").mean(), 3),
    "texts with nbsp": round(df["text"].str.contains("\xa0").mean(), 3),
    "share of apostrophes that are ʼ": round(df["text"].str.count("ʼ").sum() / max(df["text"].str.count("['’ʼ]").sum(), 1), 3),
}
uk_shape = uk.sample(200_000, random_state=0)
display(table)
display(pd.DataFrame({"uk": shape(uk_shape), "ru_sample": shape(ru), "test": shape(test), "labeling_sample": shape(sample)}))
pd.Series({f"max_len {c}": round(float((lengths["test"] <= c).mean()), 3) for c in [128, 256, 512, 700]}, name="test posts fully covered").to_frame()

,uk,ru_sample,test,labeling_sample
50%,16,18,46,36
90%,29,32,148,144
99%,41,50,361,184
max,73,201,599,187


,uk,ru_sample,test,labeling_sample
multi-line texts,0.000,0.0,0.851,0.808
texts with emoji,0.000,0.0,0.616,0.692
emoji glued to a word,0.000,0.0,0.321,0.346
texts with nbsp,0.000,0.0,0.130,0.077
share of apostrophes that are ʼ,0.003,0.0,0.178,0.143


,test posts fully covered
max_len 128,0.862
max_len 256,0.973
max_len 512,0.998
max_len 700,1.000


In [16]:
# Ceiling: gold spans -> tokens -> tags -> strings. Everything lost here is lost before any model runs.
def ceiling(df, span_col, merge_adjacent=False):
    gold, pred, bad, total = [], [], 0, 0
    for text, spans in zip(df["text"], df[span_col]):
        offsets = tokenize(text)
        tags, misaligned = spans_to_bio(offsets, spans)
        bad, total = bad + misaligned, total + len(spans)
        gold.append([text[a:b] for a, b in spans])
        pred.append(bio_to_strings(text, offsets, tags, merge_adjacent))
    return {"F1 ceiling": comp_metric(gold, pred), "misaligned spans": round(bad / max(total, 1), 4)}


uk_part = uk.sample(200_000, random_state=0)
pd.DataFrame({
    "raw markers, BIO": ceiling(uk_part, "loc_markers"),
    "raw markers, IO": ceiling(uk_part, "loc_markers", merge_adjacent=True),
    "cleaned markers, BIO": ceiling(uk_part, "spans"),
    "cleaned markers, IO": ceiling(uk_part, "spans", merge_adjacent=True),
    "labeling_sample, BIO": ceiling(sample, "spans"),
})

,"raw markers, BIO","raw markers, IO","cleaned markers, BIO","cleaned markers, IO","labeling_sample, BIO"
F1 ceiling,0.995863,0.965779,0.995813,0.978619,1.0000
misaligned spans,0.004900,0.004900,0.004300,0.004300,0.0312


In [17]:
# Which token-level signals predict LOC? Case, left and right context, on cleaned uk markers.
stats = {"case": Counter(), "left": Counter(), "right": Counter()}
loc_tok, other_tok = 0, 0
for text, spans in zip(uk_part["text"], uk_part["spans"]):
    offsets = tokenize(text)
    words = [text[a:b] for a, b in offsets]
    tags, _ = spans_to_bio(offsets, spans)
    for i, w in enumerate(words):
        if not w[0].isalpha():
            continue
        is_loc = tags[i] != "O"
        loc_tok, other_tok = loc_tok + is_loc, other_tok + (not is_loc)
        case = "ALLCAPS" if w.isupper() and len(w) > 1 else "Capitalized, sentence start" if w[0].isupper() and i == 0 else "Capitalized, mid-sentence" if w[0].isupper() else "lowercase"
        stats["case"][(case, is_loc)] += 1
        if tags[i] == "B":
            stats["left"][(words[i - 1].lower() if i else "<s>", True)] += 1
        elif not is_loc:
            stats["left"][(words[i - 1].lower() if i else "<s>", False)] += 1
        if is_loc and (i + 1 == len(tags) or tags[i + 1] != "I"):
            stats["right"][(words[i + 1].lower() if i + 1 < len(words) else "</s>", True)] += 1
        elif not is_loc:
            stats["right"][(words[i + 1].lower() if i + 1 < len(words) else "</s>", False)] += 1

base_rate = loc_tok / (loc_tok + other_tok)


def cue_table(counts, min_count=300, top=8):
    keys = {k for k, _ in counts}
    rows = [(k, counts[(k, True)], counts[(k, True)] / (counts[(k, True)] + counts[(k, False)])) for k in keys if counts[(k, True)] >= min_count]
    return pd.DataFrame(sorted(rows, key=lambda r: -r[2])[:top], columns=["cue", "LOC count", "P(LOC)"]).assign(lift=lambda d: (d["P(LOC)"] / base_rate).round(1)).round(3)


print("base rate of LOC tokens:", round(base_rate, 4))
display(cue_table(stats["case"], min_count=1))
display(cue_table(stats["left"]))
cue_table(stats["right"])

base rate of LOC tokens: 0.0287


,cue,LOC count,P(LOC),lift
0,"Capitalized, mid-sentence",68812,0.282,9.8
1,ALLCAPS,3578,0.119,4.1
2,"Capitalized, sentence start",987,0.005,0.2
3,lowercase,8041,0.003,0.1


,cue,LOC count,P(LOC),lift
0,місті,369,0.591,20.6
1,новини,326,0.556,19.4
2,території,385,0.413,14.4
3,міста,300,0.377,13.1
4,президент,541,0.326,11.4
5,президента,373,0.260,9.0
6,в,9800,0.204,7.1
7,проти,346,0.200,6.9


,cue,LOC count,P(LOC),lift
0,),682,0.118,4.1
1,(,737,0.111,3.9
2,та,2607,0.104,3.6
3,-,2445,0.077,2.7
4,—,422,0.074,2.6
5,–,746,0.073,2.5
6,.,12073,0.066,2.3
7,:,505,0.063,2.2


**Findings**
- Lengths differ: median 16 tokens in uk and 46 in test, p99 41 versus 361. A `max_len` of 512 fully covers 99.8% of test posts, so truncation can be avoided.
- 85% of test posts are multi-line and 62% contain emoji. Train has almost neither. Emoji is glued to words in 32% of posts, so the tokenizer must separate them. The apostrophe `ʼ` (18% of apostrophes in test, 0.3% in uk) needs normalising too.
- BIO versus IO: the F1 ceiling is 0.996 versus 0.979 on cleaned markers, so IO should not be used.
- Case: a capitalised word in mid-sentence is a location 28% of the time, a lowercase word 0.3%, the first word of a sentence 0.5%.
- Left context is stronger (`місті` ×20.6, `в` ×7.1) than right context (brackets, `та`, dashes ×2.5 to 4). Both directions carry signal, so a bidirectional RNN is justified.

## 6. Ideas from papers and competitions: what the data says

Each cell takes an idea from the literature and checks whether it is relevant to this data. No models are trained here.

In [18]:
# Pretrained embeddings and char layer (Lample 2016, Chiu & Nichols 2016): needed when target words are out of vocabulary.
vocab_uk = Counter(w.lower() for t in uk_part["text"] for w in TOKEN.findall(t) if w[0].isalpha())
vocab_ru = Counter(w.lower() for t in ru["text"].sample(200_000, random_state=0) for w in TOKEN.findall(t) if w[0].isalpha())
frequent = {w for w, n in vocab_uk.items() if n >= 3}
gold_words = [w.lower() for t in sample["locations"] for s in t for w in TOKEN.findall(s) if w[0].isalpha()]
test_words = [w for t in test["text"] for w in TOKEN.findall(t) if w[0].isalpha()]
names = [w for t in test["text"] for line in t.split("\n") for w in TOKEN.findall(line)[1:] if w[0].isupper()]
oov = lambda ws, v: round(float(np.mean([w.lower() not in v for w in ws])), 3)
pd.DataFrame({
    "uk vocabulary": {"all test words": oov(test_words, vocab_uk), "capitalised test words (candidates)": oov(names, vocab_uk), "gold words in sample": oov(gold_words, vocab_uk)},
    "uk, frequency >= 3": {"all test words": oov(test_words, frequent), "capitalised test words (candidates)": oov(names, frequent), "gold words in sample": oov(gold_words, frequent)},
    "uk + ru": {"all test words": oov(test_words, set(vocab_uk) | set(vocab_ru)), "capitalised test words (candidates)": oov(names, set(vocab_uk) | set(vocab_ru)), "gold words in sample": oov(gold_words, set(vocab_uk) | set(vocab_ru))},
})

,uk vocabulary,"uk, frequency >= 3",uk + ru
all test words,0.083,0.124,0.054
capitalised test words (candidates),0.157,0.225,0.129
gold words in sample,0.244,0.317,0.244


In [19]:
# Char layer (Ma & Hovy 2016, Yang 2018): do word endings identify a place among capitalised words?
capitalised = Counter()
for text, spans in zip(uk_part["text"], uk_part["spans"]):
    offsets = tokenize(text)
    tags, _ = spans_to_bio(offsets, spans)
    for i, (a, b) in enumerate(offsets[1:], 1):
        w = text[a:b]
        if w[0].isupper() and w[1:].islower() and len(w) > 4:
            capitalised[(w[-3:].lower(), tags[i] != "O")] += 1
ending_base = sum(v for (k, loc), v in capitalised.items() if loc) / sum(capitalised.values())
endings = {k for k, _ in capitalised}
rows = [(k, capitalised[(k, True)] + capitalised[(k, False)], capitalised[(k, True)] / (capitalised[(k, True)] + capitalised[(k, False)])) for k in endings if capitalised[(k, True)] + capitalised[(k, False)] >= 400]
table = pd.DataFrame(rows, columns=["last 3 letters", "words", "P(LOC)"]).sort_values("P(LOC)", ascending=False).round(3)
print(f"base rate among capitalised words: {ending_base:.3f}")
forms = Counter(t[a:b] for t, s in zip(uk["text"], uk["spans"]) for a, b in s if re.match(r"Ки[їє]в|Києв", t[a:b]) and " " not in t[a:b])
display(pd.concat([table.head(10), table.tail(5)]))
forms.most_common(8)

base rate among capitalised words: 0.315


,last 3 letters,words,P(LOC)
2,асі,680,0.990
19,їні,3373,0.982
80,їну,789,0.980
77,єві,1392,0.979
43,ьщі,503,0.948
29,ині,1653,0.946
4,иму,758,0.946
71,вці,430,0.923
70,усі,412,0.891
81,нії,1220,0.872


[('Києві', 6544),
 ('Києва', 4062),
 ('Київ', 2630),
 ('Києвом', 548),
 ('Київщині', 344),
 ('Києву', 191),
 ('Київщини', 118),
 ('Київській', 55)]

In [20]:
# Gazetteers (Chiu & Nichols; warning from Coleridge): how far does a context-free lookup of train locations go?
loc_strings = Counter(tuple(w.lower() for w in TOKEN.findall(t[a:b])) for t, s in zip(uk["text"], uk["spans"]) for a, b in s)
lexicon = {k for k, n in loc_strings.items() if n >= 5 and k}
max_words = max(map(len, lexicon))


def lookup(text, exclude_ukraine=False):
    offsets, out, i = tokenize(text), [], 0
    words = [text[a:b] for a, b in offsets]
    while i < len(words):
        for n in range(min(max_words, len(words) - i), 0, -1):
            key = tuple(w.lower() for w in words[i:i + n])
            if key in lexicon and words[i][:1].isupper():
                out.append(text[offsets[i][0]:offsets[i + n - 1][1]])
                i += n
                break
        else:
            i += 1
    return [s for s in out if not (exclude_ukraine and re.fullmatch(r"(?i)україн\w*|украин\w*", s))]


def score(y_true, y_pred):
    tp = fp = fn = 0
    for t, p in zip(y_true, y_pred):
        t, p = {hard_processing(s) for s in t}, {hard_processing(s) for s in p}
        tp, fp, fn = tp + len(t & p), fp + len(p - t), fn + len(t - p)
    return {"F1": comp_metric(y_true, y_pred), "TP": tp, "FP": fp, "FN": fn}


val_df = uk[valid_mask]
val_gold = [[t[a:b] for a, b in s] for t, s in zip(val_df["text"], val_df["spans"])]
kept_lines = lambda t: "\n".join(l for l in t.split("\n") if not is_footer(l.strip()))
y_true = sample["locations"].tolist()
print("lexicon entries:", len(lexicon))
pd.DataFrame({
    "uk is_valid": score(val_gold, [lookup(t) for t in val_df["text"]]),
    "sample": score(y_true, [lookup(t) for t in sample["text"]]),
    "sample, no footers": score(y_true, [lookup(kept_lines(t)) for t in sample["text"]]),
    "sample, no footers, no Україна": score(y_true, [lookup(kept_lines(t), True) for t in sample["text"]]),
}).T

lexicon entries: 4793


,F1,TP,FP,FN
uk is_valid,0.669240,2601.0,2037.0,534.0
sample,0.395062,16.0,33.0,16.0
"sample, no footers",0.426667,16.0,27.0,16.0
"sample, no footers, no Україна",0.450704,16.0,23.0,16.0


In [21]:
# Same-string aggregation (PII 1st): how many gold strings have several occurrences, so one confident hit is enough?
repeats = [len(re.findall(rf"(?<!\w){re.escape(s)}(?!\w)", t)) for t, gold in zip(sample["text"], sample["locations"]) for s in set(gold)]
repeated_names = []
for t in test["text"]:
    caps = Counter(w for line in t.split("\n") for w in TOKEN.findall(line)[1:] if w[0].isupper() and w[1:].islower() and len(w) > 3)
    repeated_names += list(caps.values())
pd.Series({
    "sample: gold strings": len(repeats),
    "sample: gold strings occurring more than once": int(sum(r > 1 for r in repeats)),
    "test: distinct capitalised words in a post": len(repeated_names),
    "test: of these, repeated in the same post": int(sum(r > 1 for r in repeated_names)),
})

sample: gold strings                               32
sample: gold strings occurring more than once       2
test: distinct capitalised words in a post       2738
test: of these, repeated in the same post         211
dtype: int64

In [22]:
# Multi-word and adjacent entities (CRF / BIO constraints, Lample 2016): how much structure is there to model?
def structure(df):
    c = Counter()
    for text, spans in zip(df["text"], df["spans"]):
        spans = sorted(spans)
        for (a1, b1), (a2, b2) in zip(spans, spans[1:]):
            gap = text[b1:a2]
            c["adjacent pairs, gap is whitespace"] += gap.strip() == ""
            c["adjacent pairs, gap is , or и/і/та"] += gap.strip().lower() in {",", "і", "й", "та", "и"}
        c["entities"] += len(spans)
        c["multi-token entities"] += sum(len(TOKEN.findall(text[a:b])) > 1 for a, b in spans)
    return c


pd.DataFrame({"uk": structure(uk_part), "ru_sample": structure(ru), "labeling_sample": structure(sample)}).fillna(0).astype(int)

,uk,ru_sample,labeling_sample
entities,67739,184975,32
multi-token entities,11722,28915,9
"adjacent pairs, gap is whitespace",808,2253,0
"adjacent pairs, gap is , or и/і/та",5130,15187,2


**What the data says about ideas from the literature**

| Idea | Source | What the data shows | Pointer |
|---|---|---|---|
| Pretrained fastText | Lample 2016, Chiu & Nichols 2016 | 24% of gold location words and 16% of capitalised test words are not in the uk vocabulary | try first |
| Char layer | Ma & Hovy 2016, Yang 2018 | endings `-їні`, `-єві`, `-ьщі` give P(LOC) 0.95 to 0.98, and `Київ` has more than 6 frequent forms | worth trying |
| Case feature, lowercased training copies | Chiu & Nichols, Mayhew 2019 | mid-sentence capitalisation is a strong signal, but 4 of 32 gold strings start with a lowercase letter | worth trying |
| Bidirectional RNN | Ma & Hovy 2016 | signal exists in both directions | worth trying |
| CRF or BIO constraints | Lample 2016 | multi-token entities: 17% in uk, 28% in the sample | test it |
| More streets in train | Dai & Adel 2020, Coleridge 1st | streets 1.6% versus 15.6%, and the ru boundary is wrong | worth trying, the main gap |
| Location dictionary as a rule | Chiu & Nichols, warning from Coleridge | lookup alone gives 0.67 on silver and 0.40 on the sample (16 TP, 33 FP) | only as an extra feature |
| Merging identical strings in a text | PII 1st | only 2 of 32 sample strings repeat, 8% of capitalised words in test | low priority |
| Removing channel signatures | PII 1st, post-processing | 142 test posts, removes 6 of 33 false positives on the sample | worth trying |
| Partial CRF for noisy labels | Mayhew 2019 | `Київ` is unlabelled in only 1.1% of cases and `Україна` in 9.1%, so the problem is a policy difference, not random noise | low priority |
| Pseudo-labelling | Toxic 1st, Quora 1st | test has only 477 posts and the evidence is weak | not a priority |

The sample has 32 mentions, so its numbers are indicative and not measurements.

## 7. What to take into experiments

**Data preparation (required).** Build BIO from cleaned `loc_markers`. The tokenizer should split hyphens and emoji and normalise apostrophes without changing offsets. Remove leakage, duplicates and TCH footers. Cut the output string from the original text, trim edge punctuation, and drop strings that are empty after normalisation.

**Worth trying, by decreasing expected value**
1. Pretrained fastText uk + ru.
2. A larger share of streets, districts and POI in train: existing sentences first, then mention replacement from an external street list.
3. A char layer and a case feature, training on original and lowercased text.
4. Post-like inputs: stitching sentences into multi-line texts, and predicting line by line.
5. Removing channel signatures before prediction.
6. CRF, or forbidding the `O → I` transition.
7. The `Україна` decision: by submission first, because the sample has only 4 mentions.
8. An ensemble over seeds and embeddings at the end.

**Limits of the Russian data.** Only 6.9% of test posts are in Russian and the ru street boundary is the opposite of uk, so ru should be added in limited amounts and checked separately.

**Questions for the lecturer.** Are external Telegram texts, a char language model (Flair) and CRF allowed? Manual labelling of test is forbidden by the rules.

**Reference.** In the 2025 edition the RNN baseline scored 0.562, the BERT baseline 0.648, and the best teams 0.65 to 0.66.